# The photo tint (vignette), CLAHE, and what LoFTR sees at the true place (iteration 32)

**Questions (from looking at the images).** (1) The photo has a light radial tint (bright centre, dark corners) that the map does not have: is the pipeline removing it? Default settings: the lens-vignette removal (`FLAT_FIELD_FROM_HEIGHT_M`) is OFF; what is on is the 1-99 % stretch and a whole-photo CLAHE (8x8 tiles, clip 2). (2) Photo and map visibly show the same patterns, so why does LoFTR give no match at height?

**Part A (figure, radial profile).** For 4 development photos (600, 1000, 1400, 1900 m): raw, the pipeline's version (stretch + CLAHE), vignette-removed + CLAHE, vignette-removed + local normalisation + CLAHE; and the mean grey against the distance from the frame centre for each. How much of the radial pattern is left after the pipeline's own processing?

**Part B (coarse search with cleaning).** Edge-direction ZNCC, 3 queries (headings -5, 0, +5 at scale 1.0), development photos of 500 m and higher, variants: base; vignette removal (polynomial degree 3); vignette + local normalisation (sigma 15 px); local normalisation alone; vignette without CLAHE; no CLAHE at all. Kill rule (written before the run): a variant counts only with right-within-2-sigma clearly above base (at least +5 points) above 800 m and not lower below 800 m, with no more wrong answers.

**Part C (LoFTR at the true place).** For the same 4 photos the canvas exactly as LoFTR gets it, next to the map tile at the TRUE place, with LoFTR's matches (confidence at least 0.21; green = RANSAC inliers, red = rest), for the pipeline's photo and for the cleaned photo (vignette + local normalisation).

Development photos only. Env: `EF_HEADLESS, EF_DATASET, EF_STORE, EF_LEVELS, EF_OUT`.

In [ ]:
import os, sys, math, colorsys
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("EF_LEVELS", "3,4,5,6,7,8,9,10,11").split(",")]
from geoloc import photo, queries, scoring, fine_matching, flat_field
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length, cut_map_features
from geoloc.edge_features import normalise_cells
from geoloc.levels import level_for_height, level_gsd, map_piece
from geoloc.map_store import MapStore
CELL = settings.CELL; TILE = settings.TILE
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/edge_features"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)       # the dataset folder holds the development photos only
heights = np.array([float(r["alt"]) for r in rows]); print(len(rows), "photos, heights", heights.min(), heights.max())

In [ ]:
def edge_cells(image, sigma, cell_valid=None):
    # 8-direction edge strength summed over 4x4-pixel cells, gradient blur = sigma pixels (sigma 1 = the pipeline)
    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), sigma)
    gx = np.zeros_like(blurred); gy = np.zeros_like(blurred)
    gx[:, 1:-1] = blurred[:, 2:] - blurred[:, :-2]; gy[1:-1, :] = blurred[2:, :] - blurred[:-2, :]
    strength = np.hypot(gx, gy); angle = np.arctan2(gy, gx) % np.pi; bin_id = np.minimum((angle / np.pi * settings.N_BINS).astype(int), settings.N_BINS - 1)
    r, c = image.shape[0] // CELL, image.shape[1] // CELL; out = np.zeros((settings.N_BINS, r, c), np.float32)
    for b in range(settings.N_BINS):
        out[b] = (strength * (bin_id == b))[:r * CELL, :c * CELL].reshape(r, CELL, c, CELL).sum(axis=(1, 3))
    return out

def canvas_features(canvas, valid, sigma):
    safe = cv2.erode(valid.astype(np.uint8), np.ones((7, 7), np.uint8)); side = canvas.shape[0] // CELL
    cell_valid = safe.reshape(side, CELL, side, CELL).mean(axis=(1, 3)) >= 0.8
    return normalise_cells(edge_cells(canvas, sigma), cell_valid) * cell_valid, cell_valid

def window_pixels(level_image, top, left, cells_wide):
    size = cells_wide * CELL; r0, c0 = top * CELL, left * CELL; piece = np.zeros((size, size), np.float32); h, w = level_image.shape
    a0, a1 = max(r0, 0), min(r0 + size, h); b0, b1 = max(c0, 0), min(c0 + size, w)
    if a1 > a0 and b1 > b0: piece[a0 - r0:a1 - r0, b0 - c0:b1 - c0] = np.asarray(level_image[a0:a1, b0:b1], np.float32) / 255.0
    return piece

def map_features(piece, sigma):
    feats = edge_cells(piece, sigma); has = (piece[:feats.shape[1] * CELL, :feats.shape[2] * CELL].reshape(feats.shape[1], CELL, feats.shape[2], CELL).min(axis=(1, 3)) > 0)
    has = cv2.erode(has.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    feats = normalise_cells(feats, has) * has; return feats, has

def hsv_picture(feats):
    # hue = dominant edge direction, brightness = total strength (99th percentile = full)
    total = feats.sum(axis=0); hue = feats.argmax(axis=0) / settings.N_BINS; val = np.clip(total / (np.percentile(total, 99) + 1e-6), 0, 1)
    rgb = np.zeros(total.shape + (3,), np.float32)
    for i in range(total.shape[0]):
        for j in range(total.shape[1]): rgb[i, j] = colorsys.hsv_to_rgb(hue[i, j], 0.9, val[i, j])
    return rgb

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
dev = pd.DataFrame(dict(i=range(len(rows)), height=heights))
picked = [int(dev.iloc[(dev.height - h).abs().argsort().iloc[0]].i) for h in (600, 1000, 1400, 1900)]
def with_settings(flat=None, degree=3, local=None, clahe=True):
    settings.FLAT_FIELD_FROM_HEIGHT_M = flat; settings.FLAT_FIELD_DEGREE = degree; settings.LOCAL_NORM_SIGMA = local
    if clahe: photo.normalise_frame = ORIGINAL_NORMALISE
    else:
        def stretch_only(frame):
            frame = frame.astype(np.float32); lo, hi = np.percentile(frame, [1, 99]); return np.clip((frame - lo) / (hi - lo + 1e-6), 0, 1)
        photo.normalise_frame = stretch_only
ORIGINAL_NORMALISE = photo.normalise_frame
def radial_profile(img, bins=14):
    h, w = img.shape; rr, cc = np.mgrid[0:h, 0:w]; r = np.hypot(cc - (w - 1) / 2, rr - (h - 1) / 2); r = r / r.max()
    edges = np.linspace(0, 1, bins + 1); idx = np.digitize(r.ravel(), edges) - 1
    return np.array([img.ravel()[idx == b].mean() for b in range(bins)]), (edges[:-1] + edges[1:]) / 2
fig, axes = plt.subplots(len(picked), 5, figsize=(21, 3.6 * len(picked))); report = []
for row, index in enumerate(picked):
    meta = rows[index]; camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"]); level = level_for_height(pose["height"], camera)
    raw = photo.load_photo(path).astype(np.float32); lo, hi = np.percentile(raw, [1, 99]); raw_view = np.clip((raw - lo) / (hi - lo + 1e-6), 0, 1)
    variants = {}
    with_settings(None); variants["pipeline (stretch + CLAHE)"] = photo.normalise_frame(flat_field.clean(raw, pose, camera))
    with_settings(0, 3, None); variants["vignette removed + CLAHE"] = photo.normalise_frame(flat_field.clean(raw, pose, camera))
    with_settings(0, 3, 15); variants["vignette + local norm + CLAHE"] = photo.normalise_frame(flat_field.clean(raw, pose, camera))
    with_settings(None)
    shown = [("raw (1-99 % stretch)", raw_view)] + list(variants.items())
    for col, (title, img) in enumerate(shown):
        axes[row, col].imshow(img, cmap="gray"); axes[row, col].set_title("%s\n%.0f m" % (title, pose["height"]) if col == 0 else title, fontsize=9); axes[row, col].axis("off")
    for title, img in shown:
        prof, rr = radial_profile(img); axes[row, 4].plot(rr, (prof - prof.mean()) / (img.std() + 1e-9), label=title)
        report.append(dict(photo=meta["filename"][:6], height=pose["height"], variant=title, centre_minus_corner_in_std=float((prof[:3].mean() - prof[-3:].mean()) / (img.std() + 1e-9))))
    axes[row, 4].set_title("mean grey against distance from the centre\n(in units of the picture's own std)", fontsize=9); axes[row, 4].grid(alpha=.3)
    if row == 0: axes[row, 4].legend(fontsize=7)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "tint_and_clahe.png"), dpi=90)
print(pd.DataFrame(report).round(2).to_string(index=False))

In [ ]:
# ---- Part B: coarse edge search with cleaning variants ----
VARIANTS = {"base": dict(flat=None), "vignette": dict(flat=0, degree=3), "vignette+local15": dict(flat=0, degree=3, local=15.0), "local15 only": dict(flat=0, degree=0, local=15.0),
            "vignette, no CLAHE": dict(flat=0, degree=3, clahe=False), "no CLAHE": dict(flat=None, clahe=False)}
query_list = [q for q in queries.query_list() if abs(q["scale_factor"] - 1.0) < 1e-6]; records = []
for index, meta in enumerate(rows):
    if float(meta["alt"]) < 500: continue
    meta = move_hint(meta, index, 5000.0, 0); camera = photo.read_camera(meta); pose = photo.read_pose(meta)
    prior = store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"])); truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))
    sigma_m = scoring.sigma_m(pose["height"], camera["fx"]); level = level_for_height(pose["height"], camera); path = os.path.join(DATASET, "images", meta["filename"])
    for name, kw in VARIANTS.items():
        with_settings(**kw); picture = photo.prepare_photo(path, pose, camera, level); cache = {}; best = None
        for q in query_list:
            canvas, valid = photo.draw_canvas(picture, pose, camera, level, q["heading"], q["scale_factor"])
            try: score_map, geometry = score_map_for_query(canvas, valid, level, prior, store.features, cache)
            except ValueError: continue
            usable = (score_map > -1) & geometry["inside"]; v = score_map[usable]; peak = pick_peaks(score_map, geometry, 1)[0]; z = float((peak[0] - v.mean()) / v.std())
            if best is None or z > best[0]: best = (z, math.hypot(peak[1] - truth[0], peak[2] - truth[1]) / sigma_m)
        if best: records.append(dict(photo=index, height=pose["height"], variant=name, z=best[0], err=best[1]))
    with_settings(None)
    if index % 20 == 0: print(index, flush=True)
tab = pd.DataFrame(records); tab.to_csv(os.path.join(OUT, "cleaning_coarse.csv"), index=False)
tab["band"] = pd.cut(tab.height, [500, 800, 1100, 1400, 2100])
agg = tab.groupby(["band", "variant"], observed=True).err.agg(n="size", right2=lambda e: (e <= 2).mean(), wrong=lambda e: (e > 8).mean(), within1=lambda e: (e <= 1).mean()).round(2)
for col in ["right2", "wrong", "within1"]: print(col); print(agg[col].unstack("variant")[list(VARIANTS)].to_string())
hi_ = tab[tab.height >= 800]; print("above 800 m, all bands:"); print(hi_.groupby("variant").err.agg(n="size", right2=lambda e: (e <= 2).mean(), wrong=lambda e: (e > 8).mean(), within1=lambda e: (e <= 1).mean()).round(3).loc[list(VARIANTS)].to_string())

In [ ]:
# ---- Part C: LoFTR at the true place, as the pipeline gives it ----
loftr = fine_matching.load_loftr(os.path.join(os.environ.get("EF_MODELS", os.path.join(os.path.abspath(HEADLESS), "models")), settings.LOFTR_FILE))
fig, axes = plt.subplots(len(picked), 2, figsize=(14, 5.6 * len(picked))); rep = []
for row, index in enumerate(picked):
    meta = rows[index]; camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"]); level = level_for_height(pose["height"], camera)
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"])); tile = map_piece(store.levels[level], level, truth[0], truth[1], TILE)
    for col, (title, kw) in enumerate([("pipeline photo", dict(flat=None)), ("cleaned (vignette + local norm 15)", dict(flat=0, degree=3, local=15.0))]):
        with_settings(**kw); picture = photo.prepare_photo(path, pose, camera, level); canvas, valid = photo.draw_canvas(picture, pose, camera, level, 0.0, 1.0)
        canvas_picture = fine_matching.grey_canvas(canvas, valid); pa, pb, conf = fine_matching.loftr_matches(loftr, canvas_picture, tile)
        keep = conf >= settings.LOFTR_CONFIDENCE; matrix, inl = fine_matching.fit_similarity(pa[keep], pb[keep]); n_inl = int(inl.sum()) if matrix is not None else 0
        both = np.concatenate([canvas_picture, np.ones((TILE, 6), np.float32), tile], axis=1); ax = axes[row, col]; ax.imshow(both, cmap="gray", vmin=0, vmax=1); ax.axis("off")
        pk, qk = pa[keep], pb[keep]
        for k in range(min(len(pk), 150)):
            good = matrix is not None and inl[k]; ax.plot([pk[k, 0], qk[k, 0] + TILE + 6], [pk[k, 1], qk[k, 1]], color="lime" if good else "red", lw=0.6, alpha=0.8)
        ax.set_title("%s, %.0f m: canvas | map at the TRUE place\n%d matches (conf >= %.2f), %d RANSAC inliers" % (title, pose["height"], int(keep.sum()), settings.LOFTR_CONFIDENCE, n_inl), fontsize=9)
        rep.append(dict(photo=meta["filename"][:6], height=pose["height"], variant=title, matches=int(keep.sum()), inliers=n_inl))
    with_settings(None)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "loftr_at_truth.png"), dpi=80); print(pd.DataFrame(rep).to_string(index=False))